In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from qmplot import manhattanplot
from qmplot import qqplot
import re
import sys
import subprocess
import numpy as np
from io import StringIO

from google.cloud import storage

In [ ]:
# Google Cloud file access
# reads in file from google cloud folder
def blob_as_csv(bucket, path, sep='\s+', header='infer'):
    blob = bucket.get_blob(path)
    blob = blob.download_as_bytes()
    blob = str(blob, 'utf-8')
    blob = StringIO(blob)
    df = pd.read_csv(blob, sep=sep, header=header)
    return df

# gets folders from Google Cloud
def get_gcloud_bucket(bucket_name): 
    storage_client = storage.Client(project='gp2-release-terra')
    bucket = storage_client.bucket(bucket_name, user_project='gp2-release-terra')
    return bucket


In [ ]:
# List folders to see which chromosome splits are available
def list_folders(bucket_name, prefix=""):
    """
    List all folders in a GCS bucket path.

    :param bucket_name: Name of the GCS bucket.
    :param prefix: Prefix path to search in the bucket.
    :return: List of folder names.
    """
    client = storage.Client()
    bucket = client.get_bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=prefix, delimiter='/')
    
    folders = set()
    
    for page in blobs.pages:
        folders.update(page.prefixes)  # Use prefixes to get folder names
    
    return list(folders)

In [ ]:
def plot_summary_stats(data, cohort, outcome, model, out_prefix):
  xtick = set(['chr' + i for i in list(map(str, range(1, 15))) + ['16', '18', '20', '22']])

  if model == "lmm_gallop":
    f, ax = plt.subplots(figsize=(15, 7), facecolor="w", edgecolor="k")
    manhattanplot(data=data,
                  title=f"Manhattan Intercept {cohort} {outcome}",
                  pv="Pi", ax = ax, 
                  xtick_label_set=xtick)
    plt.savefig(f"{out_prefix}_{cohort}_{outcome}_manhattan_intercept.{model}.png", dpi=300)
    f, ax = plt.subplots(figsize=(15, 7), facecolor="w", edgecolor="k")
    qqplot(data=data["Pi"],
            marker="o",
            title=f"QQ Intercept {cohort} {outcome}",
            xlabel=r"Expected -log(P)",
            ylabel=r"Observed -log(P)",
            ax=ax)
    plt.savefig(f"{out_prefix}_{cohort}_{outcome}_qq_intercept.{model}.png", dpi=300)

    f, ax = plt.subplots(figsize=(15, 7), facecolor="w", edgecolor="k")
    manhattanplot(data=data,
                  title=f"Manhattan Slope {cohort} {outcome}",
                  pv="Ps", ax = ax, 
                  xtick_label_set=xtick)
    plt.savefig(f"{out_prefix}_{cohort}_{outcome}_manhattan_slope.{model}.png", dpi=300)
    f, ax = plt.subplots(figsize=(15, 7), facecolor="w", edgecolor="k")
    qqplot(data=data["Ps"],
            marker="o",
            title=f"QQ Slope {cohort} {outcome}",
            xlabel=r"Expected -log(P)",
            ylabel=r"Observed -log(P)",
            ax=ax)
    plt.savefig(f"{out_prefix}_{cohort}_{outcome}_qq_slope.{model}.png", dpi=300)

  elif model in ["glm","cph"]:
    f, ax = plt.subplots(figsize=(15, 7), facecolor="w", edgecolor="k")
    manhattanplot(data=data,
                  title=f"Manhattan {model} {cohort} {outcome}",
                  pv="P", ax = ax,
                  xtick_label_set=xtick)
    plt.savefig(f"{out_prefix}_{cohort}_{outcome}_manhattan.{model}.png", dpi=300)
    f, ax = plt.subplots(figsize=(15, 7), facecolor="w", edgecolor="k")
    qqplot(data=data["P"],
            marker="o",
            title=f"QQ {model} {cohort} {outcome}",
            xlabel=r"Expected -log(P)",
            ylabel=r"Observed -log(P)",
            ax=ax)
    plt.savefig(f"{out_prefix}_{cohort}_{outcome}_qq.{model}.png", dpi=300)

  else:
    print("Model flag not recognised")
    return

  print("Results plotting success") 

In [ ]:
# Full file
suffix = "null"

label = 'EUR'
analysis = 'CS_disc'
month = "JAN" # JAN = R9, DEC = R7

if analysis == 'CS':
    model = 'glm'
elif analysis == 'LT':
    model = 'lmm_gallop'
elif analysis == 'SURV': 
    model = 'cph'

# pull data from different Google Cloud folders
gp2_data_bucket = get_gcloud_bucket('gp2_working_eu')

# get chromosome splits - will have to loop through ancestries and analyses
prefix = f'nicole/{month}/{label}/{analysis}/'
folders = list_folders(gp2_data_bucket, prefix)

# col names depend on model
all_chroms = pd.DataFrame()

# loop through and merge all available chromosome files
for folder in folders:
    # loop through analysis folders
    prefix = f'{folder}RESULTS/'
    results_folders = list_folders(gp2_data_bucket, prefix)

    for results_folder in results_folders:
        if "MANHATTAN" not in results_folder:
            results_path = f'{results_folder}{label}_case_control_y_allresults.tsv'
            print(results_path)

            # Prep each chromosome for plotting
            df = blob_as_csv(gp2_data_bucket, results_path, sep='\t')
            df = df.dropna(how="any", axis=0)
            df['chr_order'] = df['#CHROM'].str.replace('chr', '')
            df['chr_order'] = df['chr_order'].astype(int)

            all_chroms = pd.concat([all_chroms, df])

# needed for labeling
gwas_name = os.path.splitext(os.path.basename(results_path))[0]
cohort = gwas_name.split('_')[1]
find_phenoname = cohort + r'_(.*?)_allresults'
result = re.search(find_phenoname, gwas_name)
pheno = result.group(1)
cohort_suffix = f"{cohort}.{suffix}" if suffix != "" else cohort

display(all_chroms.chr_order.value_counts())
all_chroms.drop_duplicates(subset = ['chr_order', 'POS', 'ID'], inplace = True)
display(all_chroms.chr_order.value_counts())

# sort full dataframe
all_chroms = all_chroms.sort_values(by=['chr_order', 'POS'])

display(all_chroms.head())

In [ ]:
chrom = 12
chrom_results = all_chroms[all_chroms.chr_order == int(chrom)]
chrom_final = chrom_results[['chr_order', 'POS', 'REF', 'ALT', 'P', 'BETA', 'SE', 'A1_FREQ']] # indiv chroms

# # chrom_final = all_chroms[['chr_order', 'POS', 'REF', 'ALT', 'Ps', 'BETAs', 'SEs', 'A1_FREQ']] # R7
# chrom_final = all_chroms[['chr_order', 'POS', 'REF', 'ALT', 'P', 'BETA', 'SE', 'A1_FREQ']]

# # chrom_final.rename(columns = {'chr_order': 'CHROM', 'Ps': 'P', 'BETAs': 'BETA', 'SEs': 'SE', 'A1_FREQ': 'allele_freq'}, inplace = True) # R7
chrom_final.rename(columns = {'chr_order': 'CHROM', 'A1_FREQ': 'allele_freq'}, inplace = True)

display(chrom_final)

In [ ]:
chrom_final.to_csv(f'locuszoom/{label}_chrom{chrom}_{analysis}_results.tsv', sep = '\t', index = False) # per chrom
# chrom_final.to_csv(f'locuszoom/{label}_{analysis}_R9_all_chrom_results.tsv', sep = '\t', index = False) # R9 output